## Ucitavanje i Sredjivanje Nizova Podataka

In [1]:
#ucitavanje potrebnih biblioteka
import pandas as pd
import numpy as np

In [2]:
#uvoz podataka
#sezona 23/24
df_23_4 = pd.read_csv("super-lig-2023.csv")
#sezona 24/25
df_24_5 = pd.read_csv("super-lig-2024.csv")
#sezona 25/6
link_25 = pd.read_html("https://fixturedownload.com/results/super-lig-2025")
df_25_6 = link_25[0]


## Ukidanje Duplih Vrijednosti za Nazive Klubova

## Određivanje Duzine Trenutnoj Sezoni i Prikaz

In [3]:
#odrediti duzinu sezone - OBAVEZNO!!!
df_25_6 = df_25_6[df_25_6["Result"] != "-"]
df_25_6.tail(5)

,Round Number,Date,Location,Home Team,Away Team,Result
148,17,21/12/2025 14:30,Alanya Oba Stadium,Alanyaspor,Fatih Karagümrük,2 - 0
149,17,21/12/2025 20:00,Gürsel Aksel Stadium,Göztepe,Samsunspor,2 - 0
150,17,21/12/2025 20:00,Ali Sami Yen Spor Kompleksi,Galatasaray,Kasimpasa,3 - 0
151,17,22/12/2025 17:00,Basaksehir Fatih Terim Stadium,Istanbul Basaksehir,Gaziantep,5 - 1
152,17,22/12/2025 20:00,Eryaman Stadium,Gençlerbirligi,Trabzonspor,4 - 3


In [4]:
#sjedinjavanje ta tri df niza podataka - po osi jedan - odnosno uzduž
df_BL = pd.concat([df_23_4, df_24_5, df_25_6], axis=0)

In [5]:
df_BL["Home Team"].unique()

array(['Trabzonspor', 'Kasimpasa', 'Konyaspor', 'Kayserispor',
       'Pendikspor', 'Sivasspor', 'Adana Demirspor', 'Fenerbahçe',
       'Alanyaspor', 'Fatih Karagümrük', 'Istanbulspor', 'Antalyaspor',
       'Çaykur Rizespor', 'Hatayspor', 'Galatasaray',
       'Istanbul Basaksehir', 'Gaziantep', 'Besiktas', 'Samsunspor',
       'Ankaragücü', 'Bodrum', 'Göztepe', 'Eyüpspor', 'Kocaelispor',
       'Gençlerbirligi'], dtype=object)

In [6]:
#prikaz sjedinjenog niza podatak 
#tj. sve tri sezone u jednom nizu podataka u obliku df 
df_BL.tail()

,Match Number,Round Number,Date,Location,Home Team,Away Team,Result
148,NaN,17,21/12/2025 14:30,Alanya Oba Stadium,Alanyaspor,Fatih Karagümrük,2 - 0
149,NaN,17,21/12/2025 20:00,Gürsel Aksel Stadium,Göztepe,Samsunspor,2 - 0
150,NaN,17,21/12/2025 20:00,Ali Sami Yen Spor Kompleksi,Galatasaray,Kasimpasa,3 - 0
151,NaN,17,22/12/2025 17:00,Basaksehir Fatih Terim Stadium,Istanbul Basaksehir,Gaziantep,5 - 1
152,NaN,17,22/12/2025 20:00,Eryaman Stadium,Gençlerbirligi,Trabzonspor,4 - 3


## Predprocesovanje Niza Podataka

In [7]:
#Izbacivanje Nerelevantnih Osobina
df_BL = df_BL.drop(["Match Number","Round Number","Date","Location"],axis=1)
df_BL.shape

(873, 3)

In [8]:
df_BL.rename(columns={"Home Team":"H","Away Team":"A"},inplace=True)

In [9]:
df_BL.to_csv("t-liga.csv")
df_BL

,H,A,Result
0,Trabzonspor,Antalyaspor,1 - 0
1,Kasimpasa,Ankaragücü,3 - 2
2,Konyaspor,Istanbulspor,1 - 1
3,Kayserispor,Galatasaray,0 - 0
4,Pendikspor,Hatayspor,1 - 5
...,...,...,...
148,Alanyaspor,Fatih Karagümrük,2 - 0
149,Göztepe,Samsunspor,2 - 0
150,Galatasaray,Kasimpasa,3 - 0
151,Istanbul Basaksehir,Gaziantep,5 - 1


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

#One Hot Encoder Timova - Domacin i Gost
onehot_encoder_domacin = OneHotEncoder(handle_unknown="ignore",drop="first")  # Ensure parameters are correct
kodiran_domacin = onehot_encoder_domacin.fit_transform(df_BL[["H"]]).toarray()
df_domacin = pd.DataFrame(kodiran_domacin,columns=onehot_encoder_domacin.get_feature_names_out(["H"]))

In [11]:
df_domacin.columns

Index(['H_Alanyaspor', 'H_Ankaragücü', 'H_Antalyaspor', 'H_Besiktas',
       'H_Bodrum', 'H_Eyüpspor', 'H_Fatih Karagümrük', 'H_Fenerbahçe',
       'H_Galatasaray', 'H_Gaziantep', 'H_Gençlerbirligi', 'H_Göztepe',
       'H_Hatayspor', 'H_Istanbul Basaksehir', 'H_Istanbulspor', 'H_Kasimpasa',
       'H_Kayserispor', 'H_Kocaelispor', 'H_Konyaspor', 'H_Pendikspor',
       'H_Samsunspor', 'H_Sivasspor', 'H_Trabzonspor', 'H_Çaykur Rizespor'],
      dtype='object')

In [12]:
#One Hot Encoder Timova - Gosti
onehot_encoder_gost = OneHotEncoder(handle_unknown="ignore",drop="first")  # Ensure parameters are correct
kodiran_gost = onehot_encoder_gost.fit_transform(df_BL[["A"]]).toarray()
df_gost = pd.DataFrame(kodiran_gost,columns=onehot_encoder_gost.get_feature_names_out(["A"]))

In [13]:
#Sjedinjavanje tih osobina domacin i gost
df_jedan = pd.concat([df_domacin,df_gost],axis=1)
df_jedan

,H_Alanyaspor,H_Ankaragücü,H_Antalyaspor,H_Besiktas,H_Bodrum,H_Eyüpspor,H_Fatih Karagümrük,H_Fenerbahçe,H_Galatasaray,H_Gaziantep,...,A_Istanbulspor,A_Kasimpasa,A_Kayserispor,A_Kocaelispor,A_Konyaspor,A_Pendikspor,A_Samsunspor,A_Sivasspor,A_Trabzonspor,A_Çaykur Rizespor
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
868,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
869,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
870,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
871,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Transformisanje Rezultata u Varijablu - Y Varijable

#### Stvaranje Vrijednosti za Dvije Kolone iz Jedne Osobine/Kolone Rezultat

In [14]:
rez_dom = df_BL["Result"].str.slice(0,1)

In [15]:
rez_dom.head()

0    1
1    3
2    1
3    0
4    1
Name: Result, dtype: object

In [16]:
rez_gost = df_BL["Result"].str.slice(4,5)

In [17]:
rez_gost.head()

0    0
1    2
2    1
3    0
4    5
Name: Result, dtype: object

In [18]:
df_rezultatski = pd.concat([df_BL,rez_dom,rez_gost],axis=1)

In [19]:
#promjena naziva kolonama - spajanjem smo dobili kolone istih naziva
df_rezultatski.columns = ["H","A","Result","Rez1","Rez2"]
df_rezultatski.tail()

,H,A,Result,Rez1,Rez2
148,Alanyaspor,Fatih Karagümrük,2 - 0,2,0
149,Göztepe,Samsunspor,2 - 0,2,0
150,Galatasaray,Kasimpasa,3 - 0,3,0
151,Istanbul Basaksehir,Gaziantep,5 - 1,5,1
152,Gençlerbirligi,Trabzonspor,4 - 3,4,3


In [20]:
#spremanje df_epl varijable za obradu osobine Result u excelu
df_rezultatski.to_csv("uredjivanje_skora_vani.csv")

## Uocavanje Indeksa Sezona - za Kasnije Floatiranje

In [21]:
#df_rezultatski[0:380] #sezona 23/4 od 0:380
#df_rezultatski[380:759] #sezona 24/5 od 380:759
#df_rezultatski[760:] #sezona25/6 od 760:

## Stvaranje Kolone Skor sa Vrijednostima - Ovdje

In [22]:
# Define a function to perform the conditional logic
def calculate_value(row):
    if row['Rez1'] > row['Rez2']:
        return 1
    elif row['Rez1'] < row['Rez2']:
        return -1
    else:  # row['Column1'] == row['Column2']
        return 0

In [23]:
# Apply the function to each row in the DataFrame and assign the result to a new column
df_rezultatski['Skor'] = df_rezultatski.apply(calculate_value, axis=1)
skor = df_rezultatski.drop(['H', 'A', 'Result', 'Rez1',"Rez2"],axis=1)

### Floatiranje Vrijednosti Kolone Skor - po Sezonama

In [24]:
#sezona 23/4
skor23_4 = skor[0:380]
skor24_5 = skor[380:760]
skor25_6 = skor[760:]

In [25]:
#skor.head(3)
skor23_4["Skor"] = skor23_4["Skor"] * 0.71

C:\Users\amirb\AppData\Local\Temp\ipykernel_7592\3544292283.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  skor23_4["Skor"] = skor23_4["Skor"] * 0.71


In [26]:
skor24_5["Skor"] = skor24_5["Skor"] * 0.85

C:\Users\amirb\AppData\Local\Temp\ipykernel_7592\3136881136.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  skor24_5["Skor"] = skor24_5["Skor"] * 0.85


## Konačna kolona Skor - Sjedinjavanje Vrijednosti Sezona

In [27]:
skor = pd.concat([skor23_4, skor24_5, skor25_6], axis=0, ignore_index=True)

In [28]:
df_rezultatski = df_rezultatski.reset_index(drop=True)
skor = skor.reset_index(drop=True)

In [29]:
#provjera rezultata
finalna_provjera = pd.concat([df_rezultatski,skor],axis=1,ignore_index=True)
finalna_provjera = finalna_provjera.drop([3,4,5],axis=1)
finalna_provjera.columns = ["H","A","Rezultat","Skor"]

In [30]:
finalna_provjera[-5:-1]

,H,A,Rezultat,Skor
868,Alanyaspor,Fatih Karagümrük,2 - 0,1.0
869,Göztepe,Samsunspor,2 - 0,1.0
870,Galatasaray,Kasimpasa,3 - 0,1.0
871,Istanbul Basaksehir,Gaziantep,5 - 1,1.0


# Konačni df - Podaci su Sređeni

In [31]:
skor = skor.reset_index(drop=True)
df_jedan = df_jedan.reset_index(drop=True)

In [32]:
df_BL = pd.concat([df_jedan,skor],axis=1)

In [33]:
df_BL.head(5)

,H_Alanyaspor,H_Ankaragücü,H_Antalyaspor,H_Besiktas,H_Bodrum,H_Eyüpspor,H_Fatih Karagümrük,H_Fenerbahçe,H_Galatasaray,H_Gaziantep,...,A_Kasimpasa,A_Kayserispor,A_Kocaelispor,A_Konyaspor,A_Pendikspor,A_Samsunspor,A_Sivasspor,A_Trabzonspor,A_Çaykur Rizespor,Skor
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.71
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.71
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.71


In [34]:
#spremanje kodiranog niza podataka
df_BL.to_csv("kodirani_epl.csv")

## Razdvajanje Niza Podataka na Osobine za Predvidjanje i Ciljanu Osobinu

In [35]:
# X predvidjacke, y varijabla jeste Skor Osobina
X = df_BL.drop(["Skor"],axis=1)
y = df_BL["Skor"]

In [36]:
#razdvajanje podataka u treniranju i testirajuce nizove
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.1,random_state=42)
X_train.head()

,H_Alanyaspor,H_Ankaragücü,H_Antalyaspor,H_Besiktas,H_Bodrum,H_Eyüpspor,H_Fatih Karagümrük,H_Fenerbahçe,H_Galatasaray,H_Gaziantep,...,A_Istanbulspor,A_Kasimpasa,A_Kayserispor,A_Kocaelispor,A_Konyaspor,A_Pendikspor,A_Samsunspor,A_Sivasspor,A_Trabzonspor,A_Çaykur Rizespor
487,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
306,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
529,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
259,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
260,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [37]:
#skaliranje ovih osobina
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train

array([[-0.24073114, -0.15319288, -0.23474992, ..., -0.20948265,
        -0.23171378, -0.23775532],
       [-0.24073114, -0.15319288, -0.23474992, ..., -0.20948265,
        -0.23171378, -0.23775532],
       [-0.24073114, -0.15319288, -0.23474992, ..., -0.20948265,
        -0.23171378,  4.20600478],
       ...,
       [-0.24073114, -0.15319288, -0.23474992, ..., -0.20948265,
        -0.23171378, -0.23775532],
       [-0.24073114, -0.15319288, -0.23474992, ..., -0.20948265,
        -0.23171378, -0.23775532],
       [-0.24073114, -0.15319288, -0.23474992, ..., -0.20948265,
        -0.23171378, -0.23775532]])

In [38]:
#Spremanje tog Kodera - za Kasniju Upotrebu

with open("onehot_encoder_domacin.pkl","wb") as file:
    pickle.dump(onehot_encoder_domacin,file)
    
with open("onehot_encoder_gost.pkl","wb") as file:
    pickle.dump(onehot_encoder_gost,file)
    
with open("scaler.pkl","wb") as file:
    pickle.dump(scaler,file)
    

## Stvaranje Modela i Dobijanje Predikcije

In [39]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
#sledeca biblioteka se odnosi na stvaranje neurona iz kutije layers i alatom dense
from tensorflow.keras.layers import Dense

In [40]:
#konstrukcija modela
model=Sequential([
    Dense(16,activation="relu",input_shape=(X_train.shape[1],)), #HL1 connected to I/p - Skriveni Sloj 1 povezan sa I/p
    Dense(1,activation="tanh")
])

In [41]:
#Oblikovanje Modela 
model.compile(optimizer="adam",loss="mean_absolute_error",metrics=["mae"])
model.summary()

# Keras dokumentacija pojasnjava sta se dogodi sa razlicitim odabirom "loss" i drugih parametara

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 16)                784       
                                                                 
 dense_1 (Dense)             (None, 1)                 17        
                                                                 
Total params: 801
Trainable params: 801
Non-trainable params: 0
_________________________________________________________________


In [42]:
import datetime
#uspostavljanje Tensorboard-a, zapocinjanje treniranja
#TensorBoard sluzi samo za vizualizaciju svih logovanja koje izvrsim kod treniranja nekog modela
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

#log_dir varijabla koja osigurava da ostane zabinjezen trag svakog treniranja modela
log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

#Uspostavljanje Early Stopping - sluzi za zaustavljanje treniranja ako se vrijednost gubitaka ne smanjuje
#bitni parametri su uneseni koji sluze poboljsanju funkcije ranog zaustavljanja treniranja modela...
early_stopping_callback = EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)


In [43]:
#TRENIRANJE MODELA / Tovarenje Modela
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

history = model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=50,
    callbacks= [tensorflow_callback,early_stopping_callback]
)

Epoch 1/50
25/25 [==============================] - 3s 53ms/step - loss: 0.8177 - mae: 0.8177 - val_loss: 0.7671 - val_mae: 0.7671
Epoch 2/50
25/25 [==============================] - 1s 26ms/step - loss: 0.7514 - mae: 0.7514 - val_loss: 0.7004 - val_mae: 0.7004
Epoch 3/50
25/25 [==============================] - 0s 15ms/step - loss: 0.6904 - mae: 0.6904 - val_loss: 0.6578 - val_mae: 0.6578
Epoch 4/50
25/25 [==============================] - 1s 23ms/step - loss: 0.6394 - mae: 0.6394 - val_loss: 0.6164 - val_mae: 0.6164
Epoch 5/50
25/25 [==============================] - 0s 17ms/step - loss: 0.6038 - mae: 0.6038 - val_loss: 0.5943 - val_mae: 0.5943
Epoch 6/50
25/25 [==============================] - 0s 18ms/step - loss: 0.5769 - mae: 0.5769 - val_loss: 0.5752 - val_mae: 0.5752
Epoch 7/50
25/25 [==============================] - 0s 19ms/step - loss: 0.5548 - mae: 0.5548 - val_loss: 0.5670 - val_mae: 0.5670
Epoch 8/50
25/25 [==============================] - 0s 20ms/step - loss: 0.5365 - m

In [44]:
#Procjena Rada Modela preko test podataka
test_loss, test_mae = model.evaluate(X_test,y_test)

3/3 [==============================] - 0s 7ms/step - loss: 0.5477 - mae: 0.5477


In [45]:
#spremanje modela
model.save("t-liga_float_model.h5")

In [46]:
#moze biti da skaliranje je neophodno, onda je potrebno i obnoviti test parametara modela, te iznova
#-trenirati model i na kraju ga ponovo spremiti.

# ODREDI FORMU TIMOVA

In [47]:
#uvoz podataka
#sezona 23/24
df_23_4 = pd.read_csv("super-lig-2023.csv")
#sezona 24/25
df_24_5 = pd.read_csv("super-lig-2024.csv")
#sezona 25/6
link_25 = pd.read_html("https://fixturedownload.com/results/super-lig-2025")
df_25_6 = link_25[0]


In [48]:
#odrediti duzinu sezone - OBAVEZNO!!!
df_25_6 = df_25_6[df_25_6["Result"] != "-"]
df_25_6.tail(5)

,Round Number,Date,Location,Home Team,Away Team,Result
148,17,21/12/2025 14:30,Alanya Oba Stadium,Alanyaspor,Fatih Karagümrük,2 - 0
149,17,21/12/2025 20:00,Gürsel Aksel Stadium,Göztepe,Samsunspor,2 - 0
150,17,21/12/2025 20:00,Ali Sami Yen Spor Kompleksi,Galatasaray,Kasimpasa,3 - 0
151,17,22/12/2025 17:00,Basaksehir Fatih Terim Stadium,Istanbul Basaksehir,Gaziantep,5 - 1
152,17,22/12/2025 20:00,Eryaman Stadium,Gençlerbirligi,Trabzonspor,4 - 3


In [49]:
#sjedinjavanje ta tri df niza podataka - po osi jedan - odnosno uzduž
df_BL = pd.concat([df_23_4, df_24_5, df_25_6], axis=0)
df_BL = df_BL.drop(["Match Number","Round Number","Date","Location"],axis=1)
df_BL.rename(columns={"Home Team":"H","Away Team":"A"},inplace=True)

rez_dom = df_BL["Result"].str.slice(0,1)
rez_gost = df_BL["Result"].str.slice(4,5)

df_rezultatski = pd.concat([df_BL,rez_dom,rez_gost],axis=1)
df_rezultatski.columns = ["H","A","Result","Rez1","Rez2"]

df1 = df_rezultatski

In [50]:
#definisanje funkcije...
def klub(tim):
    # Filtriramo DataFrame
    df = df1[(df1["H"] == tim) | (df1["A"] == tim)].copy()
    
    uslovi = [
    (df['H'] == tim) & (df['Rez1'] < df["Rez2"]),
    (df['A'] == tim) & (df['Rez1'] < df["Rez2"]),
    (df['H'] == tim) & (df['Rez1'] == df["Rez2"]),
    (df['A'] == tim) & (df['Rez1'] == df["Rez2"]),
    (df['H'] == tim) & (df['Rez1'] > df["Rez2"]),
    (df['A'] == tim) & (df['Rez1'] > df["Rez2"]),]
    
    vrijednosti = [-1, 1, 0, 0, 1, -1]
    df['Forma'] = np.select(uslovi, vrijednosti, default=0)
    df = df.drop(["Rez1","Rez2"],axis=1).tail(5)
    rezultat = df["Forma"].sum() / len(df["Forma"])
    return rezultat

## Provjera Rada Te Funkcije

In [51]:
klub('Samsunspor')

-0.6

In [52]:
#izvlacenje naziva svih klubova
timovi = pd.concat([df1["H"], df1["A"]]).unique()
timovi

rezultati = {}

#petlja
for tim in timovi:
    rezultat = klub(tim)
    rezultati[tim] = rezultat

# Kreiranje DataFrame-a iz rezultata
df_forme = pd.DataFrame(list(rezultati.items()),columns = ["Klub","Rezultat"])
df_forme["Rezultat"] = df_forme["Rezultat"]

## Rezultat Forme

In [53]:
#Izracun
df_forme 

,Klub,Rezultat
0,Trabzonspor,0.4
1,Kasimpasa,-0.2
2,Konyaspor,-0.4
3,Kayserispor,0.0
4,Pendikspor,0.0
5,Sivasspor,-0.4
6,Adana Demirspor,-0.4
7,Fenerbahçe,0.6
8,Alanyaspor,0.0
9,Fatih Karagümrük,-0.6


In [54]:
#Ponovna provjera vrijednosti...
klub("Samsunspor")

-0.6

In [55]:
df_forme.to_csv("forma_h.csv")

In [56]:
df_forme["Rezultat"] = df_forme["Rezultat"] * -1

In [57]:
df_forme


,Klub,Rezultat
0,Trabzonspor,-0.4
1,Kasimpasa,0.2
2,Konyaspor,0.4
3,Kayserispor,-0.0
4,Pendikspor,-0.0
5,Sivasspor,0.4
6,Adana Demirspor,0.4
7,Fenerbahçe,-0.6
8,Alanyaspor,-0.0
9,Fatih Karagümrük,0.6


In [58]:
df_forme.to_csv("forma_a.csv")

Kraj Skripte